# Intro to NumPy & Pandas (with Telco Customer Churn data)

This notebook teaches **very simple** NumPy ideas and the **Pandas DataFrame** skills you need to explore a real CSV.

We will use: `../data/telco/WA_Fn-UseC_-Telco-Customer-Churn.csv`

**What you will learn**
1. Tiny NumPy arrays (create, shape, basic math)
2. Load a CSV into a DataFrame
3. Peek at the data (`head`, `shape`, `columns`, `info`, `describe`)
4. Select rows and columns
5. Filter rows (conditions)
6. Count values and simple group summaries
7. Find and fix missing values
8. Create / rename columns and sort

Run cells top to bottom.

## 0. Install & import (if needed)

If imports fail, run this once in a terminal:

```bash
pip install numpy pandas
```

In [ ]:
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. NumPy in 5 minutes

NumPy stores numbers in **arrays**. Pandas builds on this idea for tables (DataFrames).

In [ ]:
# Create a 1D array (a list of numbers)
tenure_sample = np.array([1, 34, 2, 45, 8])
print(tenure_sample)
print("shape:", tenure_sample.shape)   # how many items
print("dtype:", tenure_sample.dtype)   # number type

In [ ]:
# Simple math on every element at once
print("plus 1:", tenure_sample + 1)
print("times 2:", tenure_sample * 2)
print("mean:", tenure_sample.mean())
print("max:", tenure_sample.max())
print("min:", tenure_sample.min())

In [ ]:
# A small 2D array (like a mini table: rows x columns)
charges = np.array([
    [29.85, 29.85],   # MonthlyCharges, TotalCharges for customer 1
    [56.95, 1889.5],  # customer 2
    [53.85, 108.15],  # customer 3
])
print(charges)
print("shape (rows, cols):", charges.shape)
print("first row:", charges[0])
print("MonthlyCharges column:", charges[:, 0])

## 2. Load the CSV into a Pandas DataFrame

A **DataFrame** is a table: rows = customers, columns = features.

In [ ]:
from pathlib import Path

HERE = Path.cwd()
REPO = HERE.parent if (HERE / "pandas_numpy_intro.ipynb").exists() else HERE
TELCO_CSV = REPO / "data" / "telco" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(TELCO_CSV)
df

## 3. First look at the data

These are the commands you will use almost every time you open a new dataset.

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Last 3 rows
df.tail(3)

In [ ]:
print("rows, columns:", df.shape)
print("number of rows:", len(df))
print("\ncolumn names:")
print(df.columns.tolist())

In [ ]:
# Types of each column (object often means text)
df.dtypes

In [ ]:
# Compact summary: non-null counts + types
df.info()

In [ ]:
# Stats for numeric columns (count, mean, min, max, ...)
df.describe()

## 4. Select columns and rows

- One column → a **Series** (like a single list with an index)
- Several columns → another **DataFrame**

In [ ]:
# One column
df["Churn"].head()

In [ ]:
# Several columns
df[["customerID", "gender", "tenure", "MonthlyCharges", "Churn"]].head()

In [ ]:
# Rows by position (iloc): first 3 rows, columns 0 and 1
df.iloc[0:3, 0:2]

In [ ]:
# Rows/columns by label (loc)
df.loc[0:2, ["customerID", "Contract", "Churn"]]

## 5. Filter rows (conditions)

Ask questions like: *Which customers churned?* or *Who has tenure under 12 months?*

In [ ]:
# Customers who churned
churned = df[df["Churn"] == "Yes"]
print("churned customers:", len(churned))
churned.head()

In [ ]:
# Combine conditions with & (and) / | (or)
# Tip: put each condition in parentheses
new_and_churned = df[(df["tenure"] < 12) & (df["Churn"] == "Yes")]
print("tenure < 12 AND churned:", len(new_and_churned))
new_and_churned[["customerID", "tenure", "Contract", "Churn"]].head()

In [ ]:
# Filter with .isin()
fiber_or_dsl = df[df["InternetService"].isin(["Fiber optic", "DSL"])]
print("Fiber or DSL:", len(fiber_or_dsl))
fiber_or_dsl["InternetService"].value_counts()

## 6. Counts and simple group summaries

In [ ]:
# How many Yes / No for Churn?
df["Churn"].value_counts()

In [ ]:
# Same as percentages
df["Churn"].value_counts(normalize=True).round(3)

In [ ]:
# Average MonthlyCharges by Contract type
df.groupby("Contract")["MonthlyCharges"].mean().round(2)

In [ ]:
# Churn count by Contract
df.groupby("Contract")["Churn"].value_counts()

In [ ]:
# Several stats at once
df.groupby("InternetService")[["tenure", "MonthlyCharges"]].agg(["mean", "median", "count"]).round(2)

## 7. Missing values (important for this dataset)

`TotalCharges` looks numeric but is stored as text. Some values are blank spaces → they become missing after conversion.

In [ ]:
# Convert TotalCharges to numbers; bad/blank values become NaN (missing)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Count missing values per column
df.isna().sum()

In [ ]:
# Look at the rows with missing TotalCharges
missing = df[df["TotalCharges"].isna()]
print("missing TotalCharges:", len(missing))
missing[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
# Simple fix: fill missing TotalCharges with 0
# (These rows often have tenure == 0, so total charges really are ~0)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("missing after fill:", df["TotalCharges"].isna().sum())
df["TotalCharges"].describe().round(2)

## 8. Create columns, rename, sort

In [ ]:
# New column from existing ones
df["avg_charge_per_month"] = df["TotalCharges"] / df["tenure"].replace(0, np.nan)

df[["tenure", "MonthlyCharges", "TotalCharges", "avg_charge_per_month"]].head()

In [ ]:
# Map Yes/No Churn to 1/0 (useful later for models)
df["ChurnFlag"] = df["Churn"].map({"Yes": 1, "No": 0})
df[["Churn", "ChurnFlag"]].head()

In [ ]:
# Rename a column (does not change the original unless you assign back)
df_renamed = df.rename(columns={"customerID": "id", "MonthlyCharges": "monthly_bill"})
df_renamed[["id", "monthly_bill", "Churn"]].head()

In [ ]:
# Sort by MonthlyCharges (highest first)
df.sort_values("MonthlyCharges", ascending=False)[
    ["customerID", "MonthlyCharges", "Contract", "Churn"]
].head(10)

## 9. Tiny practice (try these yourself)

Uncomment and complete the lines below.

In [ ]:
# 1) How many customers are Female?
# df[df["gender"] == "???"].shape[0]

# 2) Average tenure of customers who did NOT churn
# df[df["Churn"] == "???"]["tenure"].mean()

# 3) Top 5 PaymentMethod counts
# df["PaymentMethod"].value_counts().head()

# 4) Customers with Fiber optic internet AND Month-to-month contract
# df[(df["InternetService"] == "???") & (df["Contract"] == "???")].head()

In [ ]:
# Answers (run after you try)
print("Female customers:", (df["gender"] == "Female").sum())
print("Avg tenure (no churn):", round(df[df["Churn"] == "No"]["tenure"].mean(), 2))
print("\nPaymentMethod top 5:")
print(df["PaymentMethod"].value_counts().head())
print(
    "\nFiber + month-to-month:",
    len(df[(df["InternetService"] == "Fiber optic") & (df["Contract"] == "Month-to-month")]),
)

## Cheat sheet

| Goal | Code |
|------|------|
| Load CSV | `pd.read_csv("file.csv")` |
| First rows | `df.head()` |
| Size | `df.shape` |
| Column | `df["col"]` |
| Several columns | `df[["a", "b"]]` |
| Filter | `df[df["col"] == value]` |
| Counts | `df["col"].value_counts()` |
| Group mean | `df.groupby("col")["x"].mean()` |
| Missing count | `df.isna().sum()` |
| Fill missing | `df["col"].fillna(0)` |
| Sort | `df.sort_values("col")` |
| New column | `df["new"] = ...` |

When you are comfortable with this, open `logistic_regression/main.ipynb` for the full churn classification pipeline.